# Notebook 7 — Capstone: Monte Carlo Risk + Your AI Analyst

### 📺 Brought to you by **Christian Martinez: AI for Finance**
👉 **Subscribe here: https://www.youtube.com/@christianmartinezAIforFinance**

> *Python for CFOs — a 7-part journey from spreadsheet to strategy.*

---
**Level:** 🔴 Advanced  
The grand finale. Run a Monte Carlo simulation on a budget, build a portfolio risk model, then wire in Claude to auto-write the executive commentary on your numbers.

---

## Part A — Monte Carlo: stress-testing a forecast
A single forecast is one guess. Monte Carlo runs it 10,000 times across uncertain inputs to show the *range* of outcomes.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Assumptions with uncertainty (mean, std-dev)
N = 10_000
revenue   = np.random.normal(10_000_000, 1_200_000, N)   # uncertain top line
gross_pct = np.random.normal(0.55, 0.04, N)              # uncertain margin
opex      = np.random.normal(3_200_000, 250_000, N)      # uncertain costs

ebitda = revenue*gross_pct - opex

print(f"Mean EBITDA:        ${ebitda.mean():,.0f}")
print(f"5th percentile:     ${np.percentile(ebitda,5):,.0f}  (downside)")
print(f"95th percentile:    ${np.percentile(ebitda,95):,.0f}  (upside)")
print(f"P(EBITDA < $2M):    {(ebitda<2_000_000).mean():.1%}")

In [ ]:
plt.figure(figsize=(11,4))
plt.hist(ebitda, bins=60, color="#4f81bd", alpha=0.8)
plt.axvline(ebitda.mean(), color="black", label="Mean")
plt.axvline(np.percentile(ebitda,5), color="red", linestyle="--", label="5th pct")
plt.axvline(np.percentile(ebitda,95), color="green", linestyle="--", label="95th pct")
plt.title("Monte Carlo: 10,000 EBITDA Scenarios")
plt.xlabel("EBITDA ($)"); plt.legend(); plt.tight_layout(); plt.show()

## Part B — Portfolio risk (Value at Risk)
Using `asset_returns.csv` — 2 years of daily returns for 4 assets — compute portfolio volatility and 1-day VaR.

In [ ]:
try:
    rets = pd.read_csv("asset_returns.csv", parse_dates=["date"], index_col="date")
except FileNotFoundError:
    np.random.seed(7)
    rets=pd.DataFrame(np.random.normal(0.0006,0.018,(504,4)),
        columns=["TICK_A","TICK_B","TICK_C","TICK_D"],
        index=pd.bdate_range("2024-01-01",periods=504))

weights = np.array([0.4,0.3,0.2,0.1])
port_ret = rets.dot(weights)

annual_vol = port_ret.std()*np.sqrt(252)
var_95 = np.percentile(port_ret, 5)
print(f"Annualized volatility: {annual_vol:.1%}")
print(f"1-day 95% VaR: {var_95:.2%}  → on a $10M book, ${abs(var_95)*10e6:,.0f} at risk")

## Part C — Your AI Financial Analyst 🤖
Now the magic: feed your computed numbers to Claude and have it write board-ready commentary automatically.

**Setup (one time):**
1. Get a key at console.anthropic.com  
2. In Colab, click the 🔑 (Secrets) panel on the left → add a secret named `ANTHROPIC_API_KEY`  
3. Run the cells below.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic, json
from google.colab import userdata   # Colab secrets

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# Package the numbers we computed above
financial_summary = {
    "mean_ebitda": round(float(ebitda.mean())),
    "ebitda_downside_p5": round(float(np.percentile(ebitda,5))),
    "ebitda_upside_p95": round(float(np.percentile(ebitda,95))),
    "prob_ebitda_below_2m": round(float((ebitda<2_000_000).mean()),3),
    "portfolio_annual_vol": round(float(annual_vol),3),
    "portfolio_1d_var_95": round(float(var_95),4),
}
print(json.dumps(financial_summary, indent=2))

In [ ]:
prompt = f"""You are a CFO's financial analyst. Based on this Monte Carlo
and portfolio risk output, write a concise (150-word) executive commentary
for the board. Cover: expected EBITDA, downside risk, and portfolio exposure.
Be direct and quantitative.

Data: {json.dumps(financial_summary)}"""

msg = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{"role":"user","content":prompt}]
)
print(msg.content[0].text)

### 4. Auto-generate a full board memo
Chain it: give Claude richer context and ask for a structured memo.

In [ ]:
memo_prompt = f"""Write a board memo titled 'Q4 Risk & Outlook'.
Use these figures: {json.dumps(financial_summary)}.
Structure: (1) Headline, (2) EBITDA outlook with the 5th-95th percentile range,
(3) Key risks, (4) One recommendation. Keep it under 250 words, professional tone."""

memo = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=600,
    messages=[{"role":"user","content":memo_prompt}]
)
print(memo.content[0].text)

with open("board_memo.txt","w") as f:
    f.write(memo.content[0].text)
print("\n💾 Saved board_memo.txt")

### 🎯 Your turn / Capstone challenge
Combine everything you've learned:
1. Load your own company's P&L (Notebook 2 loader)
2. Build a Monte Carlo around *your* assumptions
3. Have Claude write the commentary
4. Export a dashboard (Notebook 3) + memo into one report

That's a full **AI-augmented FP&A workflow** — something most finance teams don't have yet.

---
### ✅ You did it — Beginner to Advanced
You can now: write Python, wrangle data, visualize, analyze customers, forecast, allocate capital, model risk, and orchestrate AI. **You're a CFO who codes.**

🎓 **Keep learning — subscribe for the next series:**  
📺 **Christian Martinez: AI for Finance — https://www.youtube.com/@christianmartinezAIforFinance**